# 03 Train Video Model

Train a CNN-LSTM hybrid model for deepfake detection using a staged training approach.

### Training Phases:
1.  **Phase 1**: Freeze the CNN backbone (EfficientNet-B0) and train the LSTM and Classifier head.
2.  **Phase 2**: Unfreeze the CNN backbone and fine-tune the entire network with a lower learning rate.

### Key constraints:
- **Config-Driven**: Parameters from `video_config.yaml`.
- **Staged Dataset**: Limited to `TRAIN_SUBSET_SIZE` videos.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
import os
import cv2
import json
import yaml
import logging
from PIL import Image

# Load Config
with open("../../configs/video_config.yaml", "r") as f:
    config = yaml.safe_load(f)

IMG_SIZE = config["model"]["frame_size"]
FRAME_COUNT = config["model"]["frame_count"]
BATCH_SIZE = config["training"]["batch_size"]
TRAIN_SUBSET = config["data"]["train_subset_size"]
MODEL_PATH = Path(config["data"]["model_save_path"])
DATA_PATH = config["data"]["raw_path"]

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Dataset & Data Discovery

In [2]:
def discover_dataset_parts(base_path):
    base_path = Path(base_path)
    if not base_path.exists(): return []
    return sorted([
        base_path / d for d in os.listdir(base_path)
        if d.startswith("dfdc_train_part_") and (base_path / d).is_dir()
    ], key=lambda x: int(x.name.split('_')[-1]))

def get_video_list(parts_paths):
    video_list = []
    for part_path in parts_paths:
        metadata_path = part_path / "metadata.json"
        if not metadata_path.exists(): continue
        with open(metadata_path, "r") as f:
            metadata = json.load(f)
        for filename, info in metadata.items():
            video_path = part_path / filename
            if video_path.exists():
                video_list.append((str(video_path), 1 if info["label"] == "FAKE" else 0))
    return video_list

class VideoFrameDataset(Dataset):
    def __init__(self, video_list, num_frames=32, transform=None):
        self.video_list = video_list
        self.num_frames = num_frames
        self.transform = transform
    def __len__(self): return len(self.video_list)
    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: cap.release(); return None
        indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = Image.fromarray(frame)
                if self.transform: frame = self.transform(frame)
                frames.append(frame)
        cap.release()
        return torch.stack(frames)
    def __getitem__(self, index):
        path, label = self.video_list[index]
        frames = self._get_frames(path)
        if frames is None: frames = torch.zeros((self.num_frames, 3, IMG_SIZE, IMG_SIZE))
        return frames, torch.tensor(label, dtype=torch.float32)

## 2. Model Architecture (CNN-LSTM)

In [3]:
class DeepfakeVideoModel(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.3):
        super(DeepfakeVideoModel, self).__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        
        self.lstm = nn.LSTM(input_size=1280, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
        x = x.view(batch_size * time_steps, C, H, W)
        features = self.feature_extractor(x).view(batch_size * time_steps, -1)
        features = features.view(batch_size, time_steps, -1)
        lstm_out, (h_n, c_n) = self.lstm(features)
        return self.classifier(h_n[-1])

def set_freeze(model, freeze_backbone=True):
    for param in model.feature_extractor.parameters():
        param.requires_grad = not freeze_backbone

## 3. Training Logic

In [4]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for frames, labels in tqdm(loader, desc="Training", leave=False):
        frames, labels = frames.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for frames, labels in loader:
            frames = frames.to(device)
            outputs = model(frames)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(probs)
            all_labels.extend(labels.numpy())
    return roc_auc_score(all_labels, all_preds), accuracy_score(all_labels, np.array(all_preds) >= 0.5)

## 4. Main Loop

In [5]:
parts = discover_dataset_parts(DATA_PATH)
all_videos = get_video_list(parts)

if all_videos:
    subset_videos = all_videos[:TRAIN_SUBSET]
    train_vids, val_vids = train_test_split(subset_videos, test_size=0.15, random_state=42)
    
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(VideoFrameDataset(train_vids, num_frames=FRAME_COUNT, transform=transform), 
                              batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=0)
    val_loader = DataLoader(VideoFrameDataset(val_vids, num_frames=FRAME_COUNT, transform=transform), 
                            batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0)
    
    model = DeepfakeVideoModel(hidden_dim=config["model"]["hidden_dim"], dropout=config["model"]["dropout"]).to(device)
    criterion = nn.BCEWithLogitsLoss()
    
    best_auc = 0.0
    
    # PHASE 1: Warm-up (Backbone Frozen)
    set_freeze(model, freeze_backbone=True)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config["training"]["phase1_lr"])
    for epoch in range(config["training"]["phase1_epochs"]):
        loss = train_epoch(model, train_loader, criterion, optimizer, device)
        auc, acc = validate(model, val_loader, device)
        print(f"P1 Epoch {epoch+1}: Loss {loss:.4f} | AUC {auc:.4f} | Acc {acc:.4f}")
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), MODEL_PATH)
            
    # PHASE 2: Fine-tuning (Backbone Unfrozen)
    set_freeze(model, freeze_backbone=False)
    optimizer = optim.Adam(model.parameters(), lr=config["training"]["phase2_lr"])
    for epoch in range(config["training"]["phase2_epochs"]):
        loss = train_epoch(model, train_loader, criterion, optimizer, device)
        auc, acc = validate(model, val_loader, device)
        print(f"P2 Epoch {epoch+1}: Loss {loss:.4f} | AUC {auc:.4f} | Acc {acc:.4f}")
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), MODEL_PATH)
            print(f"--> Best Model Saved (AUC: {best_auc:.4f})")
else:
    print("No videos found to train.")